In [3]:
cd /content/drive/MyDrive/VoteRE

/content/drive/MyDrive/VoteRE


In [6]:
import re
from pathlib import Path
import pandas as pd

# =========================================================
# CONFIG
# =========================================================
SOURCE_ROOT = Path("Data")
OUTPUT_ROOT = Path("Data/3_Voters_Predictions")

PROCESSED_COLUMNS = [
    "id",
    "Tokens",
    "Subject_Entity",
    "Object_Entity",
    "Subject_Type",
    "Object_Type",
    "Subject_Start",
    "Subject_End",
    "Object_Start",
    "Object_End",
    "True_Labels",
]

PREDICTION_CANDIDATES = [
    "LLM_Prediction",
    "Final_Prediction",
    "Predicted_Relation",
    "predicted_relation",
    "prediction",
    "Prediction",
    "output_pred",
]

# =========================================================
# HELPERS
# =========================================================
def normalize_dataset_name(name: str) -> str:
    name = name.upper()
    if name == "RETACRED":
        return "ReTACRED"
    if name == "TACRED":
        return "TACRED"
    if name == "TACREV":
        return "TACREV"
    return name


def normalize_split_name(name: str) -> str:
    name = name.upper()
    if name == "DEV":
        return "dev"
    if name == "TEST":
        return "test"
    if name == "TRAIN":
        return "train"
    if name == "EXAMPLE":
        return "example"
    return name.lower()


def extract_info_from_filename(filename: str):
    """
    Example:
    PLM_ROBERTA-LARGE_TACRED_DEV_LORA_QWEEN.csv
    PLM_ROBERTA-LARGE_RETACRED_TEST_ZEROSHOT_Mistral.csv
    """
    stem = Path(filename).stem

    dataset_match = re.search(r"(TACRED|TACREV|RETACRED)", stem, re.IGNORECASE)
    split_match = re.search(r"(TRAIN|DEV|TEST|EXAMPLE)", stem, re.IGNORECASE)
    setting_match = re.search(r"(LORA|ZEROSHOT)", stem, re.IGNORECASE)

    if not dataset_match or not split_match or not setting_match:
        return None

    dataset = normalize_dataset_name(dataset_match.group(1))
    split = normalize_split_name(split_match.group(1))
    setting = setting_match.group(1).upper()

    model_match = re.search(r"(?:LORA|ZEROSHOT)_(.+)$", stem, re.IGNORECASE)
    if not model_match:
        return None

    raw_model = model_match.group(1).strip()

    model_norm_map = {
        "QWEEN": "QWEN",
        "QWEN": "QWEN",
        "MISTRAL": "MISTRAL",
        "LAMA": "LLAMA",
        "LLAMA": "LLAMA",
    }

    model_upper = raw_model.upper()
    model = model_norm_map.get(model_upper, model_upper)

    model_folder = f"{setting}_{model}"

    return {
        "dataset": dataset,
        "split": split,
        "setting": setting,
        "model": model,
        "model_folder": model_folder,
    }


def detect_prediction_column(df: pd.DataFrame):
    for col in PREDICTION_CANDIDATES:
        if col in df.columns:
            return col

    for col in df.columns:
        if "pred" in col.lower():
            return col

    return None


def refactor_file(csv_path: Path):
    info = extract_info_from_filename(csv_path.name)
    if info is None:
        print(f"Skipping (cannot parse): {csv_path}")
        return

    df = pd.read_csv(csv_path)

    pred_col = detect_prediction_column(df)
    if pred_col is None:
        print(f"Skipping (no prediction column found): {csv_path.name}")
        return

    missing_base_cols = [c for c in PROCESSED_COLUMNS if c not in df.columns]
    if missing_base_cols:
        print(f"Skipping (missing processed columns): {csv_path.name}")
        print("Missing:", missing_base_cols)
        return

    out_df = df[PROCESSED_COLUMNS].copy()
    out_df["LLM_Prediction"] = df[pred_col]

    out_dir = OUTPUT_ROOT / info["model_folder"] / info["dataset"]
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"{info['split']}.csv"
    out_df.to_csv(out_path, index=False)

    print(f"Saved: {out_path}")


# =========================================================
# RUN
# =========================================================
all_csvs = [
    p for p in sorted(SOURCE_ROOT.rglob("*.csv"))
    if OUTPUT_ROOT not in p.parents
]

print(f"Found {len(all_csvs)} CSV files")

for csv_file in all_csvs:
    refactor_file(csv_file)

print("Done.")

Found 38 CSV files
Skipping (cannot parse): Data/1_processed/ReTACRED/dev.csv
Skipping (cannot parse): Data/1_processed/ReTACRED/test.csv
Skipping (cannot parse): Data/1_processed/ReTACRED/train.csv
Skipping (cannot parse): Data/1_processed/TACRED/dev.csv
Skipping (cannot parse): Data/1_processed/TACRED/test.csv
Skipping (cannot parse): Data/1_processed/TACRED/train.csv
Skipping (cannot parse): Data/1_processed/TACREV/dev.csv
Skipping (cannot parse): Data/1_processed/TACREV/test.csv
Skipping (cannot parse): Data/PLM_BERT-LARGE_ReTACRED_DEV.csv
Skipping (cannot parse): Data/PLM_BERT-LARGE_ReTACRED_TEST.csv
Skipping (cannot parse): Data/PLM_BERT-LARGE_TACRED_DEV.csv
Skipping (cannot parse): Data/PLM_BERT-LARGE_TACRED_TEST.csv
Skipping (cannot parse): Data/PLM_BERT-LARGE_TACREV_DEV.csv
Skipping (cannot parse): Data/PLM_BERT-LARGE_TACREV_TEST.csv
Skipping (cannot parse): Data/PLM_DEBERTA-LARGE_ReTACRED_DEV.csv
Skipping (cannot parse): Data/PLM_DEBERTA-LARGE_ReTACRED_TEST.csv
Skipping (cann

In [5]:
import re
from pathlib import Path
import pandas as pd

# =========================================================
# CONFIG
# =========================================================
SOURCE_ROOT = Path("Data")
OUTPUT_ROOT = Path("Data/3_Voters_Predictions")

PROCESSED_COLUMNS = [
    "id",
    "Tokens",
    "Subject_Entity",
    "Object_Entity",
    "Subject_Type",
    "Object_Type",
    "Subject_Start",
    "Subject_End",
    "Object_Start",
    "Object_End",
    "True_Labels",
]

PREDICTION_CANDIDATES = [
    "LLM_Prediction",
    "PLM_Prediction",
    "Final_Prediction",
    "Predicted_Relation",
    "predicted_relation",
    "prediction",
    "Prediction",
    "output_pred",
    "pred",
]

# =========================================================
# HELPERS
# =========================================================
def normalize_dataset_name(name: str) -> str:
    name = name.upper()
    if name == "RETACRED":
        return "ReTACRED"
    if name == "TACRED":
        return "TACRED"
    if name == "TACREV":
        return "TACREV"
    return name

def normalize_split_name(name: str) -> str:
    name = name.upper()
    if name == "DEV":
        return "dev"
    if name == "TEST":
        return "test"
    if name == "TRAIN":
        return "train"
    if name == "EXAMPLE":
        return "example"
    return name.lower()

def normalize_model_name(raw_model: str) -> str:
    raw = raw_model.strip().upper()
    mapping = {
        "QWEEN": "QWEN",
        "QWEN": "QWEN",
        "MISTRAL": "MISTRAL",
        "LAMA": "LLAMA",
        "LLAMA": "LLAMA",
        "BERT-LARGE": "BERT_LARGE",
        "DEBERTA-LARGE": "DEBERTA_LARGE",
        "ROBERTA BASE": "ROBERTA_BASE",
        "ROBERTA-BASE": "ROBERTA_BASE",
        "ROBERTA LARGE": "ROBERTA_LARGE",
        "ROBERTA-LARGE": "ROBERTA_LARGE",
        "SPANBERT-LARGE": "SPANBERT_LARGE",
        "SPANBERT LARGE": "SPANBERT_LARGE",
    }
    return mapping.get(raw, raw.replace("-", "_").replace(" ", "_"))

def extract_info_from_filename(filename: str):
    """
    Handles examples like:
    - PLM_ROBERTA-LARGE_TACRED_DEV_LORA_QWEEN.csv
    - PLM_ROBERTA-LARGE_RETACRED_TEST_ZEROSHOT_Mistral.csv
    - PLM_BERT-LARGE_RETACRED_DEV.csv
    - PLM_DEBERTA-LARGE_TACREV_TEST.csv
    """
    stem = Path(filename).stem

    dataset_match = re.search(r"(TACRED|TACREV|RETACRED)", stem, re.IGNORECASE)
    split_match = re.search(r"(TRAIN|DEV|TEST|EXAMPLE)", stem, re.IGNORECASE)

    if not dataset_match or not split_match:
        return None

    dataset = normalize_dataset_name(dataset_match.group(1))
    split = normalize_split_name(split_match.group(1))

    # LLM-style files
    if re.search(r"(LORA|ZEROSHOT)", stem, re.IGNORECASE):
        setting_match = re.search(r"(LORA|ZEROSHOT)", stem, re.IGNORECASE)
        setting = setting_match.group(1).upper()

        model_match = re.search(r"(?:LORA|ZEROSHOT)_(.+)$", stem, re.IGNORECASE)
        if not model_match:
            return None

        raw_model = model_match.group(1).strip()
        model = normalize_model_name(raw_model)
        voter_folder = f"{setting}_{model}"

        return {
            "dataset": dataset,
            "split": split,
            "voter_folder": voter_folder,
            "kind": "LLM",
        }

    # PLM-style files
    # Example: PLM_BERT-LARGE_RETACRED_DEV.csv
    prefix_match = re.match(r"PLM_(.+?)_(TACRED|TACREV|RETACRED)_(TRAIN|DEV|TEST|EXAMPLE)$", stem, re.IGNORECASE)
    if prefix_match:
        raw_model = prefix_match.group(1).strip()
        model = normalize_model_name(raw_model)
        voter_folder = f"PLM_{model}"

        return {
            "dataset": dataset,
            "split": split,
            "voter_folder": voter_folder,
            "kind": "PLM",
        }

    return None

def detect_prediction_column(df: pd.DataFrame):
    for col in PREDICTION_CANDIDATES:
        if col in df.columns:
            return col

    for col in df.columns:
        low = col.lower()
        if "pred" in low or "relation" in low:
            return col

    return None

def refactor_file(csv_path: Path):
    info = extract_info_from_filename(csv_path.name)
    if info is None:
        print(f"Skipping (cannot parse): {csv_path.name}")
        return

    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Skipping (read error): {csv_path.name} -> {e}")
        return

    pred_col = detect_prediction_column(df)
    if pred_col is None:
        print(f"Skipping (no prediction column found): {csv_path.name}")
        return

    missing_base_cols = [c for c in PROCESSED_COLUMNS if c not in df.columns]
    if missing_base_cols:
        print(f"Skipping (missing processed columns): {csv_path.name}")
        print("Missing:", missing_base_cols)
        return

    out_df = df[PROCESSED_COLUMNS].copy()
    out_df["Prediction"] = df[pred_col]

    out_dir = OUTPUT_ROOT / info["voter_folder"] / info["dataset"]
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"{info['split']}.csv"
    out_df.to_csv(out_path, index=False)

    print(f"Saved: {out_path}")

# =========================================================
# RUN
# =========================================================
all_csvs = [
    p for p in sorted(SOURCE_ROOT.rglob("*.csv"))
    if OUTPUT_ROOT not in p.parents
]

print(f"Found {len(all_csvs)} CSV files")

for csv_file in all_csvs:
    refactor_file(csv_file)

print("Done.")

Found 97 CSV files
Skipping (cannot parse): dev.csv
Skipping (cannot parse): test.csv
Skipping (cannot parse): train.csv
Skipping (cannot parse): dev.csv
Skipping (cannot parse): test.csv
Skipping (cannot parse): train.csv
Skipping (cannot parse): dev.csv
Skipping (cannot parse): test.csv
Skipping (cannot parse): dev.csv
Skipping (cannot parse): test.csv
Skipping (cannot parse): dev.csv
Skipping (cannot parse): test.csv
Skipping (cannot parse): dev.csv
Skipping (cannot parse): test.csv
Skipping (cannot parse): dev.csv
Skipping (cannot parse): test.csv
Skipping (cannot parse): dev.csv
Skipping (cannot parse): test.csv
Skipping (cannot parse): dev.csv
Skipping (cannot parse): test.csv
Skipping (cannot parse): dev.csv
Skipping (cannot parse): test.csv
Skipping (cannot parse): dev.csv
Skipping (cannot parse): test.csv
Skipping (cannot parse): dev.csv
Skipping (cannot parse): test.csv
Skipping (cannot parse): dev.csv
Skipping (cannot parse): test.csv
Skipping (cannot parse): dev.csv
Skippin